# Project 2 - Part (c): Quanto European PDE Pricer

This notebook tests the new `qf.qEuroBSPDE` function. It prices a European quanto call / put using the 1D finite-difference PDE solver and compares the result against an independently-coded analytical Black-Scholes formula for quanto options.

### Quanto setup
* `r_d` : domestic / payoff-currency risk-free rate (used for discounting)
* `r_f` : foreign / asset-local risk-free rate (growth rate used in the asset drift)
* `q`  : asset dividend yield
* `σ_S`, `σ_Q` : asset and FX volatilities
* `ρ` : asset-FX correlation

Drift of the asset in the payoff-currency measure: `r_f - q + ρ·σ_S·σ_Q`.

Equivalently, Black-Scholes with `discRate = r_d` and `divYield' = r_d - r_f + q - ρ·σ_S·σ_Q`.

In [1]:
import sys, os
# ensure we pick up the freshly-built pyqflib
sys.path.insert(0, os.path.abspath(os.path.join('..', '..', 'pyqflib')))
import numpy as np
from math import log, sqrt, exp
from scipy.stats import norm
import qflib as qf

print('qflib version:', qf.version())
print('qEuroBSPDE available:', hasattr(qf, 'qEuroBSPDE'))

qflib version: 1.1.0-debug
qEuroBSPDE available: True


## 1. Analytical reference (pure Python)

Since parts (a) and (b) of Project 2 have not been implemented yet, we code the closed-form quanto Black-Scholes price locally. This serves as ground truth for the PDE solver.

In [2]:
def quanto_euro_bs(payoff_type, spot, strike, T, r_d, r_f, q, sigma_S, sigma_Q, rho):
    """Analytical Black-Scholes price of a European quanto option.
    payoff_type: +1 call, -1 put."""
    # effective dividend yield so that the asset drift in the domestic measure
    # equals (r_f - q + rho*sigma_S*sigma_Q) = (r_d - q_eff).
    q_eff = r_d - r_f + q - rho * sigma_S * sigma_Q
    fwd = spot * exp((r_d - q_eff) * T)
    sigT = sigma_S * sqrt(T)
    d1 = (log(fwd / strike) + 0.5 * sigT * sigT) / sigT
    d2 = d1 - sigT
    phi = payoff_type
    df = exp(-r_d * T)
    return phi * df * (fwd * norm.cdf(phi * d1) - strike * norm.cdf(phi * d2))

## 2. Market setup

In [3]:
# clear any prior market objects
qf.mktClear()

r_d = 0.04   # domestic rate
r_f = 0.02   # foreign / local rate
q   = 0.01   # dividend yield
T   = 1.0    # expiry

sigma_S = 0.25
sigma_Q = 0.15
rho     = 0.30
spot   = 100.0
strike = 100.0

# create flat yield curves
disc_yc   = qf.ycCreate(ycname='USD_DISC',   tmats=[T], vals=[r_d], valtype=0)
growth_yc = qf.ycCreate(ycname='USD_GROWTH', tmats=[T], vals=[r_f], valtype=0)
print('market list:', qf.mktList())

market list: {'YieldCurves': ['USD_DISC', 'USD_GROWTH'], 'Volatilities': []}


## 3. Single call/put sanity check

Compare `qf.qEuroBSPDE` to the analytical formula for a single ATM call and put.

In [4]:
pdeparams = {'NTIMESTEPS': 400, 'NSPOTNODES': 400, 'NSTDDEVS': 5, 'THETA': 0.5}

for payoff_type, label in [(1, 'call'), (-1, 'put')]:
    ana = quanto_euro_bs(payoff_type, spot, strike, T, r_d, r_f, q, sigma_S, sigma_Q, rho)
    pde = qf.qEuroBSPDE(payofftype=payoff_type, strike=strike, timetoexp=T, spot=spot,
                       discountcrv=disc_yc, growthcrv=growth_yc,
                       divyield=q, volatility=sigma_S,
                       fxvol=sigma_Q, correl=rho,
                       pdeparams=pdeparams)
    err = pde['Price'] - ana
    print(f"{label:4s}: analytical={ana:.6f}  pde={pde['Price']:.6f}  err={err:+.2e}")

call: analytical=10.726684  pde=10.725936  err=-7.48e-04
put : analytical=8.663160  pde=8.662411  err=-7.48e-04


## 4. Consistency with non-quanto case

If `growth_yc == disc_yc` (i.e. `r_f = r_d`) and `ρ = 0`, the quanto adjustment vanishes and the PDE should produce the same price as the plain `qf.euroBSPDE`.

In [5]:
plain_pde = qf.euroBSPDE(payofftype=1, strike=strike, timetoexp=T, spot=spot,
                        discountcrv=disc_yc, divyield=q, volatility=sigma_S,
                        pdeparams=pdeparams)
quanto_pde_same = qf.qEuroBSPDE(payofftype=1, strike=strike, timetoexp=T, spot=spot,
                              discountcrv=disc_yc, growthcrv=disc_yc,
                              divyield=q, volatility=sigma_S,
                              fxvol=0.0, correl=0.0,
                              pdeparams=pdeparams)
print(f"plain euroBSPDE        = {plain_pde['Price']:.6f}")
print(f"quanto (zero-quanto)   = {quanto_pde_same['Price']:.6f}")
print(f"diff                   = {quanto_pde_same['Price'] - plain_pde['Price']:+.2e}")

plain euroBSPDE        = 11.234828
quanto (zero-quanto)   = 11.234828
diff                   = +0.00e+00


## 5. Convergence test

Refine the grid and confirm the PDE error decays.

In [6]:
ana = quanto_euro_bs(1, spot, strike, T, r_d, r_f, q, sigma_S, sigma_Q, rho)
print(f'analytical quanto call = {ana:.6f}')
print(f"{'NT':>6s} {'NS':>6s} {'price':>12s} {'error':>12s}")
for n in [50, 100, 200, 400, 800]:
    p = {'NTIMESTEPS': n, 'NSPOTNODES': n, 'NSTDDEVS': 5, 'THETA': 0.5}
    res = qf.qEuroBSPDE(payofftype=1, strike=strike, timetoexp=T, spot=spot,
                       discountcrv=disc_yc, growthcrv=growth_yc,
                       divyield=q, volatility=sigma_S,
                       fxvol=sigma_Q, correl=rho,
                       pdeparams=p)
    print(f"{n:>6d} {n:>6d} {res['Price']:>12.6f} {res['Price']-ana:>+12.2e}")

analytical quanto call = 10.726684
    NT     NS        price        error
    50     50    10.680086    -4.66e-02
   100    100    10.714869    -1.18e-02
   200    200    10.723705    -2.98e-03
   400    400    10.725936    -7.48e-04
   800    800    10.726497    -1.88e-04


## 6. Correlation sweep

The quanto-adjusted growth rate is `r_f - q + ρ·σ_S·σ_Q`, so a higher `ρ` raises the forward and therefore the call price. Compare PDE vs. analytical across a grid of `ρ`.

In [7]:
print(f"{'rho':>6s} {'analytical':>12s} {'pde':>12s} {'err':>12s}")
for r in [-0.8, -0.4, -0.1, 0.0, 0.1, 0.4, 0.8]:
    ana = quanto_euro_bs(1, spot, strike, T, r_d, r_f, q, sigma_S, sigma_Q, r)
    res = qf.qEuroBSPDE(payofftype=1, strike=strike, timetoexp=T, spot=spot,
                      discountcrv=disc_yc, growthcrv=growth_yc,
                      divyield=q, volatility=sigma_S,
                      fxvol=sigma_Q, correl=r,
                      pdeparams=pdeparams)
    print(f"{r:>+6.2f} {ana:>12.6f} {res['Price']:>12.6f} {res['Price']-ana:>+12.2e}")

   rho   analytical          pde          err
 -0.80     8.541827     8.541033    -7.94e-04
 -0.40     9.296057     9.295282    -7.75e-04
 -0.10     9.891721     9.890958    -7.63e-04
 +0.00    10.096068    10.095309    -7.59e-04
 +0.10    10.303336    10.302581    -7.55e-04
 +0.40    10.942791    10.942046    -7.45e-04
 +0.80    11.837046    11.836312    -7.34e-04


## 7. Strike sweep

Check a range of strikes and both call / put.

In [8]:
print(f"{'K':>6s} {'type':>5s} {'analytical':>12s} {'pde':>12s} {'err':>12s}")
for K in [80, 90, 100, 110, 120]:
    for pt, lbl in [(1, 'call'), (-1, 'put')]:
        ana = quanto_euro_bs(pt, spot, K, T, r_d, r_f, q, sigma_S, sigma_Q, rho)
        res = qf.qEuroBSPDE(payofftype=pt, strike=K, timetoexp=T, spot=spot,
                          discountcrv=disc_yc, growthcrv=growth_yc,
                          divyield=q, volatility=sigma_S,
                          fxvol=sigma_Q, correl=rho,
                          pdeparams=pdeparams)
        print(f"{K:>6d} {lbl:>5s} {ana:>12.6f} {res['Price']:>12.6f} {res['Price']-ana:>+12.2e}")

     K  type   analytical          pde          err
    80  call    23.157696    23.157790    +9.40e-05
    80   put     1.878382     1.878476    +9.40e-05
    90  call    16.163181    16.163044    -1.38e-04
    90   put     4.491762     4.491624    -1.38e-04
   100  call    10.726684    10.725936    -7.48e-04
   100   put     8.663160     8.662411    -7.48e-04
   110  call     6.805005     6.804711    -2.94e-04
   110   put    14.349375    14.349081    -2.94e-04
   120  call     4.153300     4.152783    -5.17e-04
   120   put    21.305564    21.305047    -5.17e-04
